# Phase 3.3 – Modèle de Propension à l'Achat

## Objectif
Prédire la probabilité qu'un mois/région génère une vente à forte valeur (`HIGH_VALUE_SALE`), afin de concentrer les efforts marketing sur les segments les plus rentables.

## Problème
**Classification binaire** : Prédire si un mois × région aura un ratio de ventes à forte valeur supérieur à la médiane.

## Source de données
- `ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART`
- `ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS`

## Algorithmes testés
- Logistic Regression (baseline)
- Random Forest Classifier
- Gradient Boosting Classifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

print('Librairies chargées avec succès ✅')

## 1. Chargement des données

In [ ]:
# Charger le fichier CSV exporté depuis Snowflake :
# ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART
df = pd.read_csv('marketing_performance_mart.csv')

df.columns = df.columns.str.upper()
df['MONTH_DATE'] = pd.to_datetime(df['MONTH_DATE'])
df['MONTH']  = df['MONTH_DATE'].dt.month
df['YEAR']   = df['MONTH_DATE'].dt.year
df = df[df['NB_SALES'] > 0].reset_index(drop=True)

print(f'Données chargées : {df.shape[0]} observations')
df.head()

## 2. Création de la variable cible

In [ ]:
# Calcul du ratio de ventes à forte valeur
df['HIGH_VALUE_RATIO'] = df['NB_HIGH_VALUE_SALES'] / df['NB_SALES'].replace(0, np.nan)

# Variable cible binaire : 1 si ratio > médiane, 0 sinon
median_ratio = df['HIGH_VALUE_RATIO'].median()
df['TARGET'] = (df['HIGH_VALUE_RATIO'] > median_ratio).astype(int)

print(f'Médiane du ratio de ventes à forte valeur : {median_ratio:.4f}')
print(f'\nDistribution de la variable cible :')
print(df['TARGET'].value_counts())
print(f'\nRatio classe 1 : {df["TARGET"].mean():.2%}')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['HIGH_VALUE_RATIO'].hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].axvline(median_ratio, color='red', linestyle='--', label=f'Médiane ({median_ratio:.3f})')
axes[0].set_title('Distribution du ratio ventes à forte valeur')
axes[0].set_xlabel('Ratio')
axes[0].legend()

df['TARGET'].value_counts().plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Distribution variable cible')
axes[1].set_xticklabels(['Faible propension (0)', 'Forte propension (1)'], rotation=0)
axes[1].set_ylabel('Nombre d\'observations')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
le = LabelEncoder()
df['REGION_ENCODED'] = le.fit_transform(df['REGION'])

df['HAS_PROMOTION'] = (df['NB_PROMOTIONS'] > 0).astype(int)
df['HAS_CAMPAIGN']  = (df['NB_CAMPAIGNS'] > 0).astype(int)
df['SEASON'] = df['MONTH'].apply(lambda m:
    0 if m in [12, 1, 2] else
    1 if m in [3, 4, 5] else
    2 if m in [6, 7, 8] else 3
)
df['BUDGET_PER_REACH'] = np.where(
    df['TOTAL_CAMPAIGN_REACH'] > 0,
    df['TOTAL_CAMPAIGN_BUDGET'] / df['TOTAL_CAMPAIGN_REACH'],
    0
)

feature_cols = [
    'NB_PROMOTIONS', 'AVG_DISCOUNT_PERCENTAGE',
    'NB_CAMPAIGNS', 'TOTAL_CAMPAIGN_BUDGET',
    'AVG_CONVERSION_RATE', 'TOTAL_CAMPAIGN_REACH',
    'HAS_PROMOTION', 'HAS_CAMPAIGN',
    'REGION_ENCODED', 'MONTH', 'SEASON', 'BUDGET_PER_REACH'
]

X = df[feature_cols].fillna(0)
y = df['TARGET']

print(f'Features : {len(feature_cols)}')
print(f'Observations : {len(X)}')

## 4. Split Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train : {len(X_train)} | Test : {len(X_test)}')

## 5. Entraînement des modèles

In [ ]:
models = {
    'Logistic Regression (baseline)': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred      = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred       = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {'Accuracy': acc, 'AUC-ROC': auc, 'y_pred_proba': y_pred_proba}

    print(f'\n{name}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  AUC-ROC  : {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Faible', 'Forte']))

## 6. Courbes ROC

In [ ]:
plt.figure(figsize=(10, 7))
colors_roc = ['#3498db', '#2ecc71', '#e74c3c']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    auc = res['AUC-ROC']
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire (AUC = 0.5)')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC – Modèle de Propension à l\'Achat')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = max(results, key=lambda k: results[k]['AUC-ROC'])
print(f'\n✅ Meilleur modèle : {best_model_name} (AUC = {results[best_model_name]["AUC-ROC"]:.4f})')

## 7. Matrice de confusion

In [ ]:
best_model = models[best_model_name]
if 'Logistic' in best_model_name:
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Faible propension', 'Forte propension'],
            yticklabels=['Faible propension', 'Forte propension'])
plt.title(f'Matrice de Confusion – {best_model_name}')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Importance des features

In [ ]:
rf_clf = models['Random Forest']
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors_imp = ['#e74c3c' if imp > importances.median() else '#3498db' for imp in importances]
importances.plot(kind='barh', color=colors_imp)
plt.title('Importance des features – Propension à l\'Achat (Random Forest)')
plt.xlabel('Importance')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance_propensity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 features les plus importantes :')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat} : {imp:.4f}')

## 9. Interprétation et recommandations

In [ ]:
print('=== SYNTHÈSE DU MODÈLE DE PROPENSION À L\'ACHAT ===')
print(f'\nMeilleur modèle : {best_model_name}')
print(f'AUC-ROC  : {results[best_model_name]["AUC-ROC"]:.4f}')
print(f'Accuracy : {results[best_model_name]["Accuracy"]:.4f}')

print('\n=== RECOMMANDATIONS MARKETING ===')
top_features = importances.sort_values(ascending=False).head(3)
print('\nTop 3 leviers pour maximiser la propension à acheter à forte valeur :')
for i, (feat, imp) in enumerate(top_features.items(), 1):
    print(f'  {i}. {feat} (importance : {imp:.4f})')

print('\nConclusion :')
print('  Ce modèle permet de prédire les segments mois × région')
print('  à fort potentiel de ventes premium, afin de concentrer')
print('  les ressources marketing sur les cibles les plus rentables.')

In [ ]:
print('✅ Notebook purchase_propensity terminé avec succès.')